In [4]:

import os
import gc
import json
import time
import joblib
import psutil
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

# Project folders
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR
MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"
PROCESSED_DIR = BASE_DIR / "processed_data"

for folder in [MODEL_DIR, OUTPUT_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SALES_FILE = DATA_DIR / "sales_train_validation.csv"
CALENDAR_FILE = DATA_DIR / "calendar.csv"
PRICES_FILE = DATA_DIR / "sell_prices.csv"

# Configuration
HORIZON = 28
VALIDATION_DAYS = 28

# Keep CPU usage manageable
N_THREADS = 4

print("Project:", BASE_DIR)
print("Sales file exists:", SALES_FILE.exists())
print("Calendar file exists:", CALENDAR_FILE.exists())
print("Prices file exists:", PRICES_FILE.exists())

def memory_usage_gb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 3)

print(f"Current RAM usage: {memory_usage_gb():.2f} GB")


Project: c:\Users\Koushik Hazra\ai-multi-agent-supply-chain
Sales file exists: True
Calendar file exists: True
Prices file exists: True
Current RAM usage: 1.19 GB


In [5]:

# Read sales metadata and daily sales columns
sales = pd.read_csv(
    SALES_FILE,
    dtype={
        "id": "category",
        "item_id": "category",
        "dept_id": "category",
        "cat_id": "category",
        "store_id": "category",
        "state_id": "category"
    }
)

# Read calendar
calendar = pd.read_csv(
    CALENDAR_FILE,
    parse_dates=["date"],
    dtype={
        "wm_yr_wk": "int16",
        "event_name_1": "category",
        "event_type_1": "category",
        "event_name_2": "category",
        "event_type_2": "category"
    }
)

# Read sell prices
prices = pd.read_csv(
    PRICES_FILE,
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int16",
        "sell_price": "float32"
    }
)

print("Sales shape:", sales.shape)
print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)

print("RAM:", memory_usage_gb(), "GB")


Sales shape: (30490, 1919)
Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
RAM: 1.7196693420410156 GB


In [6]:

# Select a manageable baseline sample
BASELINE_SERIES = 1000

metadata_cols = [
    "id", "item_id", "dept_id",
    "cat_id", "store_id", "state_id"
]

daily_cols = [
    col for col in sales.columns
    if col.startswith("d_")
]

# Select a reproducible sample of item-store series
baseline_sales = sales.iloc[:BASELINE_SERIES].copy()

print("Baseline series:", len(baseline_sales))
print("Number of daily columns:", len(daily_cols))


Baseline series: 1000
Number of daily columns: 1913


In [7]:

baseline_long = baseline_sales.melt(
    id_vars=metadata_cols,
    value_vars=daily_cols,
    var_name="d",
    value_name="sales"
)

baseline_long["sales"] = baseline_long["sales"].astype("float32")

baseline_long = baseline_long.merge(
    calendar,
    on="d",
    how="left"
)

baseline_long = baseline_long.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

baseline_long["sell_price"] = (
    baseline_long["sell_price"].fillna(0).astype("float32")
)

baseline_long = baseline_long.sort_values(
    ["id", "date"]
).reset_index(drop=True)

print(baseline_long.head())
print("Baseline long shape:", baseline_long.shape)
print("RAM:", memory_usage_gb(), "GB")


                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   

  state_id    d  sales       date  wm_yr_wk  ... month  year  event_name_1  \
0       CA  d_1    0.0 2011-01-29     11101  ...     1  2011           NaN   
1       CA  d_2    0.0 2011-01-30     11101  ...     1  2011           NaN   
2       CA  d_3    0.0 2011-01-31     11101  ...     1  2011           NaN   
3       CA  d_4    0.0 2011-02-01     11101  ...     2  2011           NaN   
4       CA  d_5    0.0 2011-02-02     11101  ...     2  2011           NaN   

   event_type_1 event_name_2 event_type_2 snap_CA 

In [8]:

def create_features(df):
    df = df.sort_values(["id", "date"]).copy()

    grouped = df.groupby("id", observed=True)["sales"]

    df["lag_7"] = grouped.shift(7)
    df["lag_28"] = grouped.shift(28)

    shifted_sales = grouped.shift(1)

    df["rolling_mean_7"] = (
        shifted_sales.groupby(df["id"], observed=True)
        .rolling(7)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["rolling_mean_28"] = (
        shifted_sales.groupby(df["id"], observed=True)
        .rolling(28)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["rolling_std_7"] = (
        shifted_sales.groupby(df["id"], observed=True)
        .rolling(7)
        .std()
        .reset_index(level=0, drop=True)
    )

    df["day_of_week"] = df["date"].dt.dayofweek.astype("int8")
    df["day_of_month"] = df["date"].dt.day.astype("int8")
    df["month"] = df["date"].dt.month.astype("int8")
    df["year"] = df["date"].dt.year.astype("int16")
    df["week_of_year"] = (
        df["date"].dt.isocalendar().week.astype("int16")
    )

    df["is_weekend"] = (
        df["day_of_week"] >= 5
    ).astype("int8")

    return df

baseline_features = create_features(baseline_long)

baseline_features = baseline_features.dropna(
    subset=["lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28"]
).copy()

print("Feature shape:", baseline_features.shape)
print(baseline_features.head())


Feature shape: (1885000, 31)
                               id        item_id    dept_id   cat_id store_id  \
28  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
29  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
30  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
31  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
32  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   

   state_id     d  sales       date  wm_yr_wk  ... sell_price  lag_7  lag_28  \
28       CA  d_29    0.0 2011-02-26     11105  ...        0.0    0.0     0.0   
29       CA  d_30    0.0 2011-02-27     11105  ...        0.0    0.0     0.0   
30       CA  d_31    0.0 2011-02-28     11105  ...        0.0    0.0     0.0   
31       CA  d_32    0.0 2011-03-01     11105  ...        0.0    0.0     0.0   
32       CA  d_33    0.0 2011-03-02     11105  ...        0.0    0.0     0.0   

   

In [9]:

# Find the actual date corresponding to d_1886
validation_start = (
    calendar.loc[calendar["d"] == "d_1886", "date"].iloc[0]
)

print("Validation starts:", validation_start)

# Model features
FEATURES = [
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "day_of_week",
    "day_of_month",
    "month",
    "year",
    "week_of_year",
    "is_weekend",
    "sell_price"
]

# Keep only rows with complete features
model_data = baseline_features.dropna(
    subset=FEATURES + ["sales"]
).copy()

# Add calendar category columns as model features
for col in [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]:
    if col in model_data.columns:
        model_data[col] = (
            model_data[col].astype("category")
        )
        FEATURES.append(col)

# Use date-based split
train_data = model_data[
    model_data["date"] < validation_start
].copy()

valid_data = model_data[
    model_data["date"] >= validation_start
].copy()

print("Train rows:", len(train_data))
print("Validation rows:", len(valid_data))


Validation starts: 2016-03-28 00:00:00
Train rows: 1857000
Validation rows: 28000


In [10]:

X_train = train_data[FEATURES]
y_train = train_data["sales"]

X_valid = valid_data[FEATURES]
y_valid = valid_data["sales"]

categorical_features = [
    col for col in FEATURES
    if str(X_train[col].dtype) == "category"
]

baseline_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=N_THREADS,
    verbosity=-1
)

baseline_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

joblib.dump(
    baseline_model,
    MODEL_DIR / "baseline_model.pkl"
)

print("Baseline model trained and saved.")


Training until validation scores don't improve for 50 rounds
[50]	valid_0's l2: 3.79824
[100]	valid_0's l2: 3.74077
[150]	valid_0's l2: 3.743
Early stopping, best iteration is:
[105]	valid_0's l2: 3.73816
Baseline model trained and saved.


In [11]:

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)

    denominator = np.abs(actual).sum()

    wape = (
        np.abs(actual - predicted).sum() / denominator * 100
        if denominator > 0 else np.nan
    )

    bias = (
        (predicted - actual).sum() / denominator * 100
        if denominator > 0 else np.nan
    )

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE_percent": wape,
        "Bias_percent": bias
    }

baseline_predictions = baseline_model.predict(
    X_valid,
    num_iteration=baseline_model.best_iteration_
)

baseline_metrics = calculate_metrics(
    y_valid,
    baseline_predictions
)

print(pd.Series(baseline_metrics))


MAE              0.945622
RMSE             1.933433
WAPE_percent    81.308883
Bias_percent    -1.477659
dtype: float64


In [12]:

baseline_metrics_df = pd.DataFrame([
    {"Model": "Baseline LightGBM", **baseline_metrics}
])

baseline_metrics_df.to_csv(
    OUTPUT_DIR / "baseline_performance.csv",
    index=False
)

print(baseline_metrics_df)


               Model       MAE      RMSE  WAPE_percent  Bias_percent
0  Baseline LightGBM  0.945622  1.933433     81.308883     -1.477659


In [13]:

# Clear baseline data before full processing
del baseline_features, baseline_long
del baseline_sales, model_data, train_data, valid_data
del X_train, X_valid, y_train, y_valid
del baseline_predictions

gc.collect()

# Use the latest 393 available sales days
FULL_WINDOW = 393
BATCH_SIZE = 200

all_daily_cols = [
    col for col in sales.columns
    if col.startswith("d_")
]

selected_days = all_daily_cols[-FULL_WINDOW:]

# Recreate the output folder
FULL_DIR = PROCESSED_DIR / "full"
FULL_DIR.mkdir(parents=True, exist_ok=True)

# Remove old batch files to avoid mixing runs
for file in FULL_DIR.glob("*.parquet"):
    file.unlink()

print("Selected days:", selected_days[0], "to", selected_days[-1])
print("Number of days:", len(selected_days))
print("RAM:", memory_usage_gb(), "GB")


Selected days: d_1521 to d_1913
Number of days: 393
RAM: 0.8797683715820312 GB


In [18]:

def process_series_batch(batch, batch_number):

    long_df = batch.melt(
        id_vars=metadata_cols,
        value_vars=selected_days,
        var_name="d",
        value_name="sales"
    )

    long_df["sales"] = long_df["sales"].astype("float32")

    # Merge calendar
    long_df = long_df.merge(
        calendar,
        on="d",
        how="left"
    )

    # Merge selling prices
    long_df = long_df.merge(
        prices,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left"
    )

    long_df["sell_price"] = (
        long_df["sell_price"].fillna(0).astype("float32")
    )

    long_df = long_df.sort_values(
        ["id", "date"]
    ).reset_index(drop=True)

    # Generate features
    featured = create_features(long_df)

    # Remove rows without sufficient historical data
    featured = featured.dropna(
        subset=[
            "lag_7",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_28"
        ]
    ).copy()

    # IMPORTANT: Keep the entire selected date window.
    # Do NOT filter from validation_start here.

    output_file = FULL_DIR / f"batch_{batch_number:04d}.parquet"

    featured.to_parquet(
        output_file,
        index=False,
        compression="snappy"
    )

    del long_df, featured
    gc.collect()

    return output_file


In [15]:

# Load the processed batches
full_data = pd.read_parquet(FULL_DIR)

print("Full processed shape:", full_data.shape)
print("RAM:", memory_usage_gb(), "GB")

# Ensure event categories are consistent
event_features = [
    col for col in [
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]
    if col in full_data.columns
]

for col in event_features:
    full_data[col] = full_data[col].astype("category")

# Training and validation split
full_train = full_data[
    full_data["date"] < validation_start
].copy()

full_valid = full_data[
    full_data["date"] >= validation_start
].copy()

# Use the same feature list as the baseline
print("Full training rows:", len(full_train))
print("Full validation rows:", len(full_valid))


Full processed shape: (853720, 31)
RAM: 1.4574661254882812 GB
Full training rows: 0
Full validation rows: 853720


In [16]:

X_full_train = full_train[FEATURES]
y_full_train = full_train["sales"]

X_full_valid = full_valid[FEATURES]
y_full_valid = full_valid["sales"]

full_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.04,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    max_bin=63,
    random_state=42,
    n_jobs=N_THREADS,
    verbosity=-1
)

full_model.fit(
    X_full_train,
    y_full_train,
    eval_set=[(X_full_valid, y_full_valid)],
    categorical_feature=event_features,
    callbacks=[
        lgb.early_stopping(75),
        lgb.log_evaluation(100)
    ]
)

print("Full-data model trained.")
print("Best iteration:", full_model.best_iteration_)


ValueError: Input data must be 2 dimensional and non empty.

In [17]:
print("Full data shape:", full_data.shape)
print("Training shape:", full_train.shape)
print("Validation shape:", full_valid.shape)

print("X_full_train shape:", X_full_train.shape)
print("y_full_train shape:", y_full_train.shape)

print("Date range:")
print(full_data["date"].min())
print(full_data["date"].max())

Full data shape: (853720, 31)
Training shape: (0, 31)
Validation shape: (853720, 31)
X_full_train shape: (0, 16)
y_full_train shape: (0,)
Date range:
2016-03-28 00:00:00
2016-04-24 00:00:00


In [19]:
for file in FULL_DIR.glob("*.parquet"):
    file.unlink()

batch_files = []

for batch_number, start in enumerate(
    tqdm(range(0, len(sales), BATCH_SIZE))
):

    batch = sales.iloc[
        start:start + BATCH_SIZE
    ].copy()

    output_file = process_series_batch(
        batch,
        batch_number
    )

    batch_files.append(output_file)

    del batch
    gc.collect()

print("All batches processed:", len(batch_files))

  0%|          | 0/153 [00:00<?, ?it/s]

All batches processed: 153


In [20]:
full_data = pd.concat(
    [
        pd.read_parquet(file)
        for file in batch_files
    ],
    ignore_index=True
)

full_train = full_data[
    full_data["date"] < validation_start
].copy()

full_valid = full_data[
    full_data["date"] >= validation_start
].copy()

print("Full data:", full_data.shape)
print("Training:", full_train.shape)
print("Validation:", full_valid.shape)

print("Date range:")
print(full_data["date"].min(), "to", full_data["date"].max())

MemoryError: Unable to allocate 127. MiB for an array with shape (3, 11128850) and data type float32

In [21]:
import gc

# Remove large DataFrames left over from previous attempts
for name in [
    "full_data",
    "full_train",
    "full_valid",
    "X_full_train",
    "X_full_valid",
    "y_full_train",
    "y_full_valid"
]:
    if name in globals():
        del globals()[name]

gc.collect()

print(f"RAM currently used: {memory_usage_gb():.2f} GB")

RAM currently used: 4.55 GB


In [22]:

# Keep only the columns required for model training
REQUIRED_COLUMNS = list(dict.fromkeys(
    ["id", "date", "sales"] + FEATURES
))

sampled_batches = []

SAMPLE_FRACTION = 0.05

for file in tqdm(batch_files, desc="Loading sampled batches"):

    batch = pd.read_parquet(
        file,
        columns=REQUIRED_COLUMNS
    )

    # Select complete item-store series, not random individual rows
    available_ids = batch["id"].drop_duplicates()

    number_to_sample = max(
        1,
        int(len(available_ids) * SAMPLE_FRACTION)
    )

    selected_ids = available_ids.sample(
        n=min(number_to_sample, len(available_ids)),
        random_state=42
    )

    batch = batch[
        batch["id"].isin(selected_ids)
    ].copy()

    # Use consistent event categories
    for col in event_features:
        if col in batch.columns and col in calendar.columns:
            batch[col] = pd.Categorical(
                batch[col],
                categories=calendar[col].cat.categories
            )

    sampled_batches.append(batch)

    del batch
    gc.collect()

print("Number of sampled batches:", len(sampled_batches))


Loading sampled batches:   0%|          | 0/153 [00:00<?, ?it/s]

Number of sampled batches: 153


In [23]:

full_data = pd.concat(
    sampled_batches,
    ignore_index=True
)

del sampled_batches
gc.collect()

print("Sampled data shape:", full_data.shape)
print("Number of series:", full_data["id"].nunique())
print("RAM usage:", memory_usage_gb(), "GB")


Sampled data shape: (556260, 19)
Number of series: 1524
RAM usage: 4.62176513671875 GB


In [25]:

full_train = full_data[
    full_data["date"] < validation_start
]

full_valid = full_data[
    full_data["date"] >= validation_start
]

X_full_train = full_train[FEATURES]
y_full_train = full_train["sales"]

X_full_valid = full_valid[FEATURES]
y_full_valid = full_valid["sales"]

print("Training:", X_full_train.shape)
print("Validation:", X_full_valid.shape)
print("Training date range:", full_train["date"].min(),
      "to", full_train["date"].max())
print("Validation date range:", full_valid["date"].min(),
      "to", full_valid["date"].max())
```

Check that both training and validation contain rows before proceeding.

## Step 5 — Train the model

<CodeBlock language="python" editable>
full_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=700,
    learning_rate=0.04,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    max_bin=63,
    random_state=42,
    n_jobs=4,
    verbosity=-1
)

full_model.fit(
    X_full_train,
    y_full_train,
    eval_set=[(X_full_valid, y_full_valid)],
    categorical_feature=event_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

print("Training completed!")
print("Best iteration:", full_model.best_iteration_)


SyntaxError: invalid syntax (810040739.py, line 21)